# 32 candidates: merge the Colab and local parts into a 6,000-step training set (local PC)

Runs on the local PC (Windows) with the **brio-local** kernel from `BRIO_32_candidates_local.ipynb`. No GPU is needed.

Generation was stopped early, so training uses a **subset of the train split**:

| Part | Articles | Count | File in `liputan6_candidates_32/canonical/` |
|---|---|---|---|
| Colab | 0 to 34,700 | 34,701 | `train.out` (downloaded from Drive) |
| Local PC | 132,584 to 193,882 | 61,299 | `train.132584-193883.out` |
| **Total** | | **96,000** | 96,000 ÷ 16 per step = **6,000 steps** in 1 epoch |

Steps:
1. Build `train.source`, `train.target` and `train.out` for exactly those articles, in the same order, plus `train.subset_ranges.json` (the ranges, for the paper).
2. Preprocess train (subset), val and test (full) into the BRIO json format.
3. Zip the result as `liputan6_32_6000.zip`, to upload to `MyDrive/BRIO/Dataset/` for training on Colab.

Before running: the local generation must be finished, and `train.out` must be Colab's **finished** file (34,701 articles). The settings cell checks both.

Disk space: about 3 to 5 GB for the json files plus the zip.

In [1]:
# Settings, paths and helpers. The notebook lives in colab_vsc/, the scripts in the project folder.
import os, sys, subprocess, shutil, json, random
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "colab_vsc" else Path.cwd()
os.chdir(REPO)

NUM_CANDS = 32
N_TRAIN = 193883
COLAB_RANGE = (0, 34701)          # Colab's part, end exclusive (last article 34,700)
LOCAL_RANGE = (132584, 193883)    # local PC's part, end exclusive (last article 193,882)
EXPECTED = 96000                  # 6,000 steps x 16 articles per step
N_EVAL = 10972                    # canonical val and test

DATA_DIR = REPO / "liputan6_converted" / "canonical"          # full source/target files
CAND_DIR = REPO / "liputan6_candidates_32" / "canonical"      # 32-candidate files
COLAB_FILE = CAND_DIR / "train.out"
LOCAL_FILE = CAND_DIR / "train.132584-193883.out"
SUBSET_DIR = REPO / "liputan6_converted_6000" / "canonical"   # input of preprocess.py
OUT_ROOT = REPO / "liputan6_32_6000"                          # BRIO json files (unzips to ./liputan6 on Colab)
ZIP_BASE = REPO / "liputan6_32_6000"                          # -> liputan6_32_6000.zip

# Child processes: UTF-8 for files and output. preprocess.py opens files without an encoding,
# which on Windows would mean cp1252; PYTHONUTF8=1 makes it UTF-8, as on Colab.
ENV = dict(os.environ, PYTHONUTF8="1", PYTHONIOENCODING="utf-8")

def run_py(args):
    # run a script with this kernel's Python and print its output as it comes
    proc = subprocess.Popen([sys.executable, "-u"] + [str(a) for a in args], stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace", env=ENV)
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError("%s stopped with exit code %d" % (args[0], proc.returncode))

def count_lines(path):
    with open(path, encoding="utf-8") as f:
        return sum(1 for _ in f)

print("project folder:", REPO)
ok = True
for name, path, need in [("Colab part", COLAB_FILE, COLAB_RANGE[1] - COLAB_RANGE[0]),
                         ("local part", LOCAL_FILE, LOCAL_RANGE[1] - LOCAL_RANGE[0]),
                         ("val.out", CAND_DIR / "val.out", N_EVAL), ("test.out", CAND_DIR / "test.out", N_EVAL)]:
    if not path.exists():
        print("MISSING   %-10s %s" % (name, path)); ok = False; continue
    n = count_lines(path) / NUM_CANDS
    good = n == need or (name == "Colab part" and n > need)
    ok &= good
    print("%s %-10s %9.2f of %d articles" % ("ok       " if good else "NOT READY", name, n, need))
for split in ["train", "val", "test"]:
    for ext in ["source", "target"]:
        p = DATA_DIR / (split + "." + ext)
        if not p.exists():
            print("MISSING  ", p); ok = False
print("\nready" if ok else "\nnot ready: finish the generation / download Colab's finished train.out first")

project folder: d:\project\BRIO
ok        Colab part  34701.00 of 34701 articles
ok        local part  61299.00 of 61299 articles
ok        val.out     10972.00 of 10972 articles
ok        test.out    10972.00 of 10972 articles

ready


In [2]:
# NLTK sentence splitter for preprocess.py: 'punkt' on older NLTK, 'punkt_tab' on NLTK 3.8.2 and later.
import nltk
for pkg in ["punkt", "punkt_tab"]:
    try:
        nltk.download(pkg, quiet=True)
    except Exception as e:
        print("could not download", pkg, e)
from nltk import sent_tokenize
print(sent_tokenize("Liputan6.com, Jakarta: Ini kalimat pertama. Ini kalimat kedua."))

['Liputan6.com, Jakarta: Ini kalimat pertama.', 'Ini kalimat kedua.']


## 1. Build the training subset

Stops with an error, and writes nothing, if a part is incomplete, the ranges overlap, or the total is not 96,000. If Colab's `train.out` holds more than 34,701 articles, add `"--allow_extra"` to the list below to use only its first 34,701.

In [3]:
run_py(["build_train_subset.py", "--src_dir", DATA_DIR, "--split", "train",
        "--num_cands", NUM_CANDS, "--expected_articles", EXPECTED,
        "--part", "%d:%d:%s" % (COLAB_RANGE[0], COLAB_RANGE[1], COLAB_FILE),
        "--part", "%d:%d:%s" % (LOCAL_RANGE[0], LOCAL_RANGE[1], LOCAL_FILE),
        "--out_dir", SUBSET_DIR])
print((SUBSET_DIR / "train.subset_ranges.json").read_text(encoding="utf-8"))

d:\project\BRIO\liputan6_candidates_32\canonical\train.out             articles       0-  34700: 1110432 lines, need 1110432
d:\project\BRIO\liputan6_candidates_32\canonical\train.132584-193883.out articles  132584- 193882: 1961568 lines, need 1961568
Wrote 96000 articles (3072000 candidate lines) to d:\project\BRIO\liputan6_converted_6000\canonical
Ranges: 0-34700, 132584-193882
One epoch at 16 articles per step: 6000 full optimizer steps (0 articles left over)
{
 "split": "train",
 "num_cands": 32,
 "articles": 96000,
 "split_articles": 193883,
 "ranges": [
  {
   "start": 0,
   "end_exclusive": 34701,
   "articles": 34701,
   "file": "train.out"
  },
  {
   "start": 132584,
   "end_exclusive": 193883,
   "articles": 61299,
   "file": "train.132584-193883.out"
  }
 ]
}


In [4]:
# Spot check: the first candidate of a few articles from each part, next to its reference.
# The candidate should be about the same story as the reference; a shift by one article would show here.
with open(SUBSET_DIR / "train.target", encoding="utf-8") as f:
    refs = [l.rstrip("\n") for l in f]
with open(SUBSET_DIR / "train.out", encoding="utf-8") as f:
    cands = [l.rstrip("\n") for l in f]
n_colab = COLAB_RANGE[1] - COLAB_RANGE[0]
random.seed(0)
for i in [0, n_colab - 1, n_colab, len(refs) - 1] + random.sample(range(len(refs)), 2):
    orig = COLAB_RANGE[0] + i if i < n_colab else LOCAL_RANGE[0] + (i - n_colab)
    print("subset #%d = train article %d" % (i, orig))
    print("  reference:", refs[i][:200])
    print("  candidate:", cands[i * NUM_CANDS][:200])
del cands

subset #0 = train article 0
  reference: Menurut Presiden Susilo Bambang Yudhoyono , kemiskinan kini menjadi masalah serius yang dihadapi bangsa-bangsa di Asia dan Afrika . Diharapkan masalah ini menjadi agenda penting dalam KTT Asia-Afrika 
  candidate:  presiden yudhoyono menekankan bahwa tantangan terbesar yang dihadapi bangsa-bangsa asia dan afrika saat ini adalah masalah kemiskinan yang sangat buruk . ktt asia-afrika akan berlangsung selama dua h
subset #34700 = train article 34700
  reference: Ratusan mahasiswa Universitas Jambi mendatangi Kantor Kepolisian Daerah Jambi . Mereka meminta kepala kepolisian setempat mengusut kasus pemukulan oleh sejumlah anggota Brimob terhadap seorang mahasis
  candidate:  ratusan mahasiswa universitas jambi berunjuk rasa ke kantor kepolisian daerah jambi . mereka meminta kepala kepolisian setempat mengusut kasus pemukulan oleh sejumlah anggota brimob terhadap seorang 
subset #34701 = train article 132584
  reference: Pemkab Mojokerto mengaku menga

In [5]:
# ROUGE = 0 report for the subset (same check as for the other candidate files).
run_py(["check_candidates.py", "--out_file", SUBSET_DIR / "train.out", "--tgt_file", SUBSET_DIR / "train.target",
        "--num_cands", NUM_CANDS, "--show", "1"])

File: d:\project\BRIO\liputan6_converted_6000\canonical\train.out
Articles: 96000, candidates per article: 32
Candidates with ROUGE = 0: 2574 of 3072000 (0.08%)
Articles with more than 1 such candidates: 243 (0.25%)
Distinct candidates per article (average): 29.1 of 32
Articles by number of ROUGE = 0 candidates: [(0, 94444), (1, 1313), (2, 58), (3, 47), (4, 34), (5, 19), (6, 21), (7, 20), (8, 5), (9, 8), (10, 10), (11, 1), (12, 5), (13, 5), (14, 2), (15, 2), (16, 3), (17, 1), (18, 2)]
ROUGE = 0 candidates by position (0 = first beam group): [(0, 114), (1, 118), (2, 119), (3, 126), (4, 107), (5, 105), (6, 100), (7, 87), (8, 104), (9, 85), (10, 62), (11, 75), (12, 79), (13, 70), (14, 89), (15, 73), (16, 80), (17, 76), (18, 63), (19, 65), (20, 59), (21, 70), (22, 59), (23, 55), (24, 58), (25, 41), (26, 72), (27, 62), (28, 65), (29, 73), (30, 77), (31, 86)]

Article 469, ROUGE = 0 at positions [2, 19, 29]
  reference: suasana ruang sidang pn jakpus ricuh setelah hakim memutuskan menunda si

## 2. Preprocess into the BRIO json format

`preprocess.py` scores every candidate with ROUGE ((R-1 + R-2 + R-Lsum) / 3) and writes one json file per article to `liputan6_32_6000/canonical/<split>/<n>.json`, with the keys `article`, `abstract`, `candidates` (text and score), and their `_untok` copies. This is the format `main.py` reads.

val and test stay complete (10,972 articles each), with their 32-candidate files. The `.tokenized` copies are plain copies, as in `BRIO_32_candidates.ipynb` (no Stanford tokenizer for Indonesian). The train split takes the longest (96,000 articles, 8 worker processes).

In [6]:
for split in ["val", "test"]:
    for ext in ["source", "target"]:
        shutil.copyfile(DATA_DIR / (split + "." + ext), SUBSET_DIR / (split + "." + ext))
    shutil.copyfile(CAND_DIR / (split + ".out"), SUBSET_DIR / (split + ".out"))
for split in ["train", "val", "test"]:
    for ext in ["source", "target", "out"]:
        shutil.copyfile(SUBSET_DIR / (split + "." + ext), SUBSET_DIR / (split + "." + ext + ".tokenized"))
    print(split, count_lines(SUBSET_DIR / (split + ".source")), "articles,",
          count_lines(SUBSET_DIR / (split + ".out")), "candidate lines")

train 96000 articles, 3072000 candidate lines
val 10972 articles, 351104 candidate lines
test 10972 articles, 351104 candidate lines


In [7]:
# Build liputan6_32_6000/canonical/{train,val,test}. A rerun overwrites the same file names.
for split in ["test", "val", "train"]:
    print("=====", split)
    run_py(["preprocess.py", "--src_dir", SUBSET_DIR, "--tgt_dir", OUT_ROOT / "canonical", "--split", split,
            "--cand_num", NUM_CANDS, "--dataset", "liputan6", "-l"])

# test.source / test.target are needed by main.py -e (same layout as the Colab notebook)
shutil.copyfile(DATA_DIR / "test.source", OUT_ROOT / "canonical" / "test.source")
shutil.copyfile(DATA_DIR / "test.target", OUT_ROOT / "canonical" / "test.target")
shutil.copyfile(DATA_DIR / "test.target", OUT_ROOT / "test.target")
shutil.copyfile(SUBSET_DIR / "train.subset_ranges.json", OUT_ROOT / "canonical" / "train.subset_ranges.json")

===== test

100%|██████████| 10972/10972 [01:07<00:00, 163.52it/s]
finish
===== val

100%|██████████| 10972/10972 [01:09<00:00, 157.71it/s]
finish
===== train

100%|██████████| 96000/96000 [10:17<00:00, 155.49it/s]
finish


WindowsPath('d:/project/BRIO/liputan6_32_6000/canonical/train.subset_ranges.json')

In [8]:
# Check the BRIO json files: one file per article, 32 scored candidates each, and the
# reference in each file matching the line of the same article in the subset files.
def norm(s):
    return " ".join(s.lower().split())

problems = []
for split in ["train", "val", "test"]:
    folder = OUT_ROOT / "canonical" / split
    with open(SUBSET_DIR / (split + ".target"), encoding="utf-8") as f:
        refs = [l.rstrip("\n") for l in f]
    n_files = len(os.listdir(folder))
    if n_files != len(refs):
        problems.append("%s: %d json files, %d articles" % (split, n_files, len(refs)))
    random.seed(1)
    for i in [0, len(refs) - 1] + random.sample(range(len(refs)), 20):
        with open(folder / ("%d.json" % i), encoding="utf-8") as f:
            d = json.load(f)
        missing = {"article", "abstract", "candidates", "article_untok", "abstract_untok", "candidates_untok"} - set(d)
        if missing:
            problems.append("%s/%d.json misses %s" % (split, i, missing))
            continue
        if len(d["candidates"]) != NUM_CANDS or len(d["candidates_untok"]) != NUM_CANDS:
            problems.append("%s/%d.json has %d candidates" % (split, i, len(d["candidates"])))
        if not all(0 <= c[1] <= 1 for c in d["candidates"]):
            problems.append("%s/%d.json has a candidate score outside 0-1" % (split, i))
        if norm(" ".join(d["abstract_untok"])) != norm(refs[i]):
            problems.append("%s/%d.json: reference does not match line %d of %s.target" % (split, i, i, split))
    print("%-5s %d json files, sample checked" % (split, n_files))

print("\nall checks passed" if not problems else "\nPROBLEMS:\n  " + "\n  ".join(problems[:20]))
with open(OUT_ROOT / "canonical" / "train" / "0.json", encoding="utf-8") as f:
    d = json.load(f)
print("\ntrain/0.json - best and worst candidate score: %.3f / %.3f" % (max(c[1] for c in d["candidates"]), min(c[1] for c in d["candidates"])))

train 96000 json files, sample checked
val   10972 json files, sample checked
test  10972 json files, sample checked

all checks passed

train/0.json - best and worst candidate score: 0.541 / 0.216


## 3. Zip for Colab

The zip holds `canonical/...` and `test.target` at its top level, like `liputan6_32.zip`, so it unzips into `./liputan6` on Colab. Zipping about 118,000 files takes a while.

In [9]:
zip_path = shutil.make_archive(str(ZIP_BASE), "zip", root_dir=OUT_ROOT)
print("written:", zip_path, "(%.2f GB)" % (os.path.getsize(zip_path) / 1e9))

written: d:\project\BRIO\liputan6_32_6000.zip (0.21 GB)


## 4. Training on Colab

1. Upload `liputan6_32_6000.zip` to `MyDrive/BRIO/Dataset/`.
2. In `BRIO_32_candidates.ipynb`, set `DATASET_ZIP = "/content/drive/MyDrive/BRIO/Dataset/liputan6_32_6000.zip"`, change the restore cell's `ZipFile("./liputan6_32.zip", ...)` to `liputan6_32_6000.zip`, and train with `--max_num 32`.

With 96,000 articles and the default batch size 2 × accumulate 8, one epoch is exactly 6,000 optimizer steps, and validation runs at 2,000, 4,000 and 6,000.

For the paper: the subset is articles 0–34,700 and 132,584–193,882 of the canonical train split (96,000 of 193,883, in `train.subset_ranges.json`). Articles 13,488–13,531 were generated in bfloat16 on Colab, and if the local run switched to bfloat16 partway, note the article where it switched; all other candidates are full precision.